# Week 2 Day 3 — LangGraph

## Task 1: Graph Concepts & State Design

### LangGraph Core Concepts

**StateGraph:** StateGraph is used to create and manage the overall workflow.

**Nodes:** Nodes are individual steps of the workflow. For example, Plan, Retrieve, Generate, and Critique are nodes.

**Edges:** Edges connect one node to another and define the flow of the graph.

**Conditional Edges:** Conditional edges choose the next node based on a condition. For example, if the answer quality is low, the workflow goes back to Generate.

**State:** State stores the information that is shared and updated between different nodes of the workflow.

### State Schema

For this workflow, the state will contain:

- **question:** The user's question.
- **plan:** The plan created for answering the question.
- **retrieved_info:** Information collected for the answer.
- **answer:** The generated answer.
- **critique:** Feedback about the answer quality.
- **retry_count:** Number of times the answer has been regenerated.
- **approved:** Human approval status.

### Graph Diagram

START
 ↓
PLAN
 ↓
RETRIEVE
  ↓
GENERATE
  ↓
CRITIQUE
  ↓
Quality?
 ↙       ↘
NO       YES
 ↓         ↓
GENERATE  HUMAN APPROVAL
        ↓
           FINISH

# Task 2: Build a Linear Graph


In [3]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

In [4]:
# Task 2: Define the shared state of the graph
class State(TypedDict):
    question: str
    plan: str
    retrieved_info: str
    answer: str

In [5]:
# Task 2: Create the plan node
def plan(state):
    # Create a simple plan for answering the question
    state["plan"] = "Understand the question and prepare a simple answer."
    
    # Print the updated state after the plan node
    print("After Plan:")
    print(state)
    
    return state

In [6]:
# Task 2: Retrieve information
def retrieve(state):
    # Add information to the state
    state["retrieved_info"] = "Python is a programming language."

    # Show the updated state
    print("After Retrieve:")
    print(state)

    return state

In [7]:
# Task 2: Generate the answer
def generate(state):
    # Create an answer
    state["answer"] = "Python is a programming language used for data analysis, AI, and web development."

    # Show the updated state
    print("After Generate:")
    print(state)

    return state

In [8]:
# Task 2: Create the graph
graph = StateGraph(State)

# Add the nodes
graph.add_node("plan", plan)
graph.add_node("retrieve", retrieve)
graph.add_node("generate", generate)

# Connect the nodes
graph.add_edge(START, "plan")
graph.add_edge("plan", "retrieve")
graph.add_edge("retrieve", "generate")
graph.add_edge("generate", END)

In [9]:
# Task 2: Compile the graph
app = graph.compile()

In [10]:
# Task 2: Run the graph with a sample question
result = app.invoke({
    "question": "What is Python?",
    "plan": "",
    "retrieved_info": "",
    "answer": ""
})

# Show the final state
print("Final State:")
print(result)

After Plan:
{'question': 'What is Python?', 'plan': 'Understand the question and prepare a simple answer.', 'retrieved_info': '', 'answer': ''}
After Retrieve:
{'question': 'What is Python?', 'plan': 'Understand the question and prepare a simple answer.', 'retrieved_info': 'Python is a programming language.', 'answer': ''}
After Generate:
{'question': 'What is Python?', 'plan': 'Understand the question and prepare a simple answer.', 'retrieved_info': 'Python is a programming language.', 'answer': 'Python is a programming language used for data analysis, AI, and web development.'}
Final State:
{'question': 'What is Python?', 'plan': 'Understand the question and prepare a simple answer.', 'retrieved_info': 'Python is a programming language.', 'answer': 'Python is a programming language used for data analysis, AI, and web development.'}


## Task 3: Conditional Edges and Self-Correction Loop

In this task, we add a critique step to the workflow. The generated answer is checked for quality.

If the answer is not good enough, the workflow goes back to the generate step and tries again. If the answer is good enough, the workflow finishes.

A retry counter is also added to prevent the workflow from running forever.

The workflow is:

START → PLAN → RETRIEVE → GENERATE → CRITIQUE → QUALITY CHECK

If quality is poor → GENERATE AGAIN → CRITIQUE

If quality is good → END

LangGraph makes this type of branching and loop-back workflow easy to represent using conditional edges and cycles.

In [18]:
# Task 3: Define the state for the conditional workflow

from typing import TypedDict
from langgraph.graph import StateGraph, START, END


class State3(TypedDict):
    question: str
    plan: str
    retrieved_info: str
    answer: str
    critique: str
    retry_count: int

In [25]:
# Task 3: Create the plan node

def plan3(state):
    state["plan"] = "Understand the question and prepare a simple answer."

    print("After Plan:")
    print(state)

    return state

In [31]:
# Task 3: Retrieve information

def retrieve3(state):
    state["retrieved_info"] = "Python is a programming language."

    print("After Retrieve:")
    print(state)

    return state

In [36]:
# Task 3: Generate the answer

def generate3(state):
    # First attempt gives a short answer
    if state["retry_count"] == 0:
        state["answer"] = "Python is a programming language."
    else:
        # Retry gives a better answer
        state["answer"] = "Python is a programming language used for data analysis, artificial intelligence, web development, and automation."

    print("After Generate:")
    print(state)

    return state

In [40]:
# Task 3: Critique the answer

def critique3(state):
    # Check if the answer is long enough
    if len(state["answer"]) < 50:
        state["critique"] = "Poor - The answer is too short. Try again."
    else:
        state["critique"] = "Good - The answer is clear and detailed."

    print("After Critique:")
    print(state)

    return state

In [43]:
# Task 3: Decide whether to retry or finish

def check_quality3(state):
    if state["critique"].startswith("Good"):
        return "finish"
    else:
        return "retry"

In [45]:
# Task 3: Retry the generation

def generate_again3(state):
    # Increase the retry counter
    state["retry_count"] += 1

    print("Retrying...")
    print("Retry Count:", state["retry_count"])

    return state

In [46]:
# Task 3: Create the conditional graph

graph3 = StateGraph(State3)

# Add the nodes
graph3.add_node("plan", plan3)
graph3.add_node("retrieve", retrieve3)
graph3.add_node("generate", generate3)
graph3.add_node("critique", critique3)
graph3.add_node("generate_again", generate_again3)

# Connect the main workflow
graph3.add_edge(START, "plan")
graph3.add_edge("plan", "retrieve")
graph3.add_edge("retrieve", "generate")
graph3.add_edge("generate", "critique")

# Add the conditional loop
graph3.add_conditional_edges(
    "critique",
    check_quality3,
    {
        "retry": "generate_again",
        "finish": END
    }
)

# Connect retry back to generate
graph3.add_edge("generate_again", "generate")

In [47]:
# Task 3: Compile the graph

app3 = graph3.compile()

In [48]:
# Task 3: Run the graph

result3 = app3.invoke({
    "question": "What is Python?",
    "plan": "",
    "retrieved_info": "",
    "answer": "",
    "critique": "",
    "retry_count": 0
})

print("Final State:")
print(result3)

After Plan:
{'question': 'What is Python?', 'plan': 'Understand the question and prepare a simple answer.', 'retrieved_info': '', 'answer': '', 'critique': '', 'retry_count': 0}
After Retrieve:
{'question': 'What is Python?', 'plan': 'Understand the question and prepare a simple answer.', 'retrieved_info': 'Python is a programming language.', 'answer': '', 'critique': '', 'retry_count': 0}
After Generate:
{'question': 'What is Python?', 'plan': 'Understand the question and prepare a simple answer.', 'retrieved_info': 'Python is a programming language.', 'answer': 'Python is a programming language.', 'critique': '', 'retry_count': 0}
After Critique:
{'question': 'What is Python?', 'plan': 'Understand the question and prepare a simple answer.', 'retrieved_info': 'Python is a programming language.', 'answer': 'Python is a programming language.', 'critique': 'Poor - The answer is too short. Try again.', 'retry_count': 0}
Retrying...
Retry Count: 1
After Generate:
{'question': 'What is Pyth

## Task 4: Human-in-the-Loop (HITL)

In this task, we add a human approval step to the workflow.

The workflow will pause before completing an important action and wait for human approval.

If the human approves, the workflow continues and finishes.

If the human rejects the action, the workflow can be corrected or stopped.

Human-in-the-loop is useful when an action is risky, important, or needs human judgment.

The workflow is:

START → PLAN → RETRIEVE → GENERATE → CRITIQUE → HUMAN APPROVAL → END

This shows how LangGraph can pause a workflow and resume it after human approval.

In [49]:
# Task 4: Import the required libraries

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import interrupt, Command

In [50]:
# Task 4: Define the state

class State4(TypedDict):
    question: str
    answer: str
    approval: str

In [52]:
# Task 4: Generate the answer

def generate4(state):
    state["answer"] = "Python is a programming language used for data analysis and AI."

    print("Generated Answer:")
    print(state["answer"])

    return state

In [53]:
# Task 4: Ask for human approval

def human_approval4(state):
    # Pause the workflow and wait for human approval
    approval = interrupt(
        "Do you approve this answer? Type yes or no."
    )

    state["approval"] = approval

    print("Human Approval:")
    print(state["approval"])

    return state

In [54]:
# Task 4: Check human approval

def check_approval4(state):
    if state["approval"].lower() == "yes":
        return "approved"
    else:
        return "rejected"

In [55]:
# Task 4: Create the human-in-the-loop graph

graph4 = StateGraph(State4)

# Add the nodes
graph4.add_node("generate", generate4)
graph4.add_node("human_approval", human_approval4)

# Connect the nodes
graph4.add_edge(START, "generate")
graph4.add_edge("generate", "human_approval")

# Add the approval decision
graph4.add_conditional_edges(
    "human_approval",
    check_approval4,
    {
        "approved": END,
        "rejected": END
    }
)

In [56]:
# Task 4: Add memory for pausing and resuming the workflow

memory4 = MemorySaver()

app4 = graph4.compile(
    checkpointer=memory4
)

In [57]:
# Task 4: Start the workflow

config4 = {
    "configurable": {
        "thread_id": "task4"
    }
}

result4 = app4.invoke(
    {
        "question": "What is Python?",
        "answer": "",
        "approval": ""
    },
    config4
)

print("Workflow paused for human approval.")

Generated Answer:
Python is a programming language used for data analysis and AI.
Workflow paused for human approval.


In [58]:
# Task 4: Resume the workflow after human approval

result4 = app4.invoke(
    Command(resume="yes"),
    config4
)

print("Workflow resumed.")
print("Final State:")
print(result4)

Human Approval:
yes
Workflow resumed.
Final State:
{'question': 'What is Python?', 'answer': 'Python is a programming language used for data analysis and AI.', 'approval': 'yes'}


In [70]:
# Task 4: Test human rejection

config4_no = {
    "configurable": {
        "thread_id": "task4_no"
    }
}

result4_no = app4.invoke(
    {
        "question": "What is Python?",
        "answer": "",
        "approval": ""
    },
    config4_no
)

print("Workflow paused for human approval.")

Generated Answer:
Python is a programming language used for data analysis and AI.
Workflow paused for human approval.


In [60]:
# Task 4: Resume the workflow after rejection

result4_no = app4.invoke(
    Command(resume="no"),
    config4_no
)

print("Workflow resumed.")
print("Final State:")
print(result4_no)

Human Approval:
no
Workflow resumed.
Final State:
{'question': 'What is Python?', 'answer': 'Python is a programming language used for data analysis and AI.', 'approval': 'no'}


In [61]:
# Task 5: Import MemorySaver

from langgraph.checkpoint.memory import MemorySaver

In [62]:
# Task 5: Define the state

class State5(TypedDict):
    question: str
    answer: str

In [63]:
# Task 5: Create the answer node

def answer5(state):
    state["answer"] = "Python is a programming language used for data analysis and AI."

    print("Answer:")
    print(state["answer"])

    return state

In [64]:
# Task 5: Create the graph

graph5 = StateGraph(State5)

graph5.add_node("answer", answer5)

graph5.add_edge(START, "answer")
graph5.add_edge("answer", END)

In [65]:
# Task 5: Add memory to the graph

memory5 = MemorySaver()

app5 = graph5.compile(
    checkpointer=memory5
)

In [66]:
# Task 5: Create a thread ID

config5 = {
    "configurable": {
        "thread_id": "python_conversation"
    }
}

In [67]:
# Task 5: Run the workflow

result5 = app5.invoke(
    {
        "question": "What is Python?",
        "answer": ""
    },
    config5
)

print("Final State:")
print(result5)

Answer:
Python is a programming language used for data analysis and AI.
Final State:
{'question': 'What is Python?', 'answer': 'Python is a programming language used for data analysis and AI.'}


In [68]:
# Task 5: Check the saved state

saved_state = app5.get_state(config5)

print("Saved State:")
print(saved_state.values)

Saved State:
{'question': 'What is Python?', 'answer': 'Python is a programming language used for data analysis and AI.'}


In [69]:
# Task 5: Check the state history

history = app5.get_state_history(config5)

for state in history:
    print(state.values)

{'question': 'What is Python?', 'answer': 'Python is a programming language used for data analysis and AI.'}
{'question': 'What is Python?', 'answer': ''}
{}


## Final Workflow Diagram (Mermaid)

In [ ]:
START
  ↓
PLAN
  ↓
RETRIEVE
  ↓
GENERATE
  ↓
CRITIQUE
  ↓
Quality Check
  ↙          ↘
Poor        Good
 ↓            ↓
GENERATE     HUMAN APPROVAL
AGAIN          ↓
 ↓          ┌──┴──┐
CRITIQUE   YES    NO
              ↓     ↓
         END   END